In [2]:
import pandas as pd
import os

## Somatic Mutation Variables

### GBM

In [3]:
GBM_CBIO_PATH = r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Somatic Mutations\cBioPort_GBM.txt"
GBM_MAF_PATH = r"C:\Users\abbyn\OneDrive\Desktop\Research Project\TCGA\TCGA-GBM\Somatic Mutation"

In [4]:
def get_top_drivers(path, top_n=6):
    df = pd.read_csv(path, sep="\t")
    df.columns = df.columns.str.strip()
    df["Freq"] = df["Freq"].str.replace("%", "").astype(float)

    drivers = df[df["Is Cancer Gene (source: OncoKB)"] == "Yes"].head(top_n)
    return drivers[["Gene", "Freq"]]

gbm_top_info = get_top_drivers(GBM_CBIO_PATH)
cbio_top = gbm_top_info["Gene"].tolist()

In [5]:
maf_list = []
for root, _, files in os.walk(GBM_MAF_PATH):
    for f in [f for f in files if f.endswith(".maf.gz")]:
        df = pd.read_csv(os.path.join(root, f), sep="\t", comment="#", usecols=["Tumor_Sample_Barcode", "Hugo_Symbol"])
        df["patient_id"] = df["Tumor_Sample_Barcode"].str[:12]
        maf_list.append(df[["patient_id", "Hugo_Symbol"]])

maf_df = pd.concat(maf_list, ignore_index=True).drop_duplicates()

In [6]:
mutation_matrix = maf_df.assign(value=1).pivot_table(
    index="patient_id",
    columns="Hugo_Symbol",
    values="value",
    aggfunc="max",
    fill_value=0
)

mutation_final = mutation_matrix.reindex(columns=cbio_top, fill_value=0).astype(int)

In [7]:
local_freqs = (mutation_final.sum() / len(mutation_final) * 100).round(2)

print(f"{'Gene':<8} | {'cBioPortal %':<15} | {'Local %':<10}")
print("-" * 40)
for _, row in gbm_top_info.iterrows():
    gene = row["Gene"]
    print(f"{gene:<8} | {row['Freq']:<15} | {local_freqs[gene]:<10}")

Gene     | cBioPortal %    | Local %   
----------------------------------------
TP53     | 32.1            | 31.81     
PTEN     | 33.4            | 33.15     
EGFR     | 22.5            | 22.91     
NF1      | 10.3            | 11.05     
ATRX     | 9.3             | 9.43      
PIK3CA   | 9.5             | 9.7       


### LGG

In [8]:
LGG_CBIO_PATH = r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Somatic Mutations\cBioPort_LGG.txt"
LGG_MAF_PATH = r"C:\Users\abbyn\OneDrive\Desktop\Research Project\TCGA\TCGA-LGG\Somatic Mutation"

In [9]:
lgg_top_info = get_top_drivers(LGG_CBIO_PATH)
cbio_top = lgg_top_info["Gene"].tolist()

In [10]:
maf_list = []
for root, _, files in os.walk(LGG_MAF_PATH):
    for f in [f for f in files if f.endswith(".maf.gz")]:
        df = pd.read_csv(os.path.join(root, f), sep="\t", comment="#", usecols=["Tumor_Sample_Barcode", "Hugo_Symbol"])
        df["patient_id"] = df["Tumor_Sample_Barcode"].str[:12]
        maf_list.append(df[["patient_id", "Hugo_Symbol"]])

maf_df = pd.concat(maf_list, ignore_index=True).drop_duplicates()

In [11]:
mutation_matrix = maf_df.assign(value=1).pivot_table(
    index="patient_id",
    columns="Hugo_Symbol",
    values="value",
    aggfunc="max",
    fill_value=0
)

mutation_final = mutation_matrix.reindex(columns=cbio_top, fill_value=0).astype(int)

In [12]:
local_freqs = (mutation_final.sum() / len(mutation_final) * 100).round(2)

print(f"{'Gene':<8} | {'cBioPortal %':<15} | {'Local %':<10}")
print("-" * 40)
for _, row in lgg_top_info.iterrows():
    gene = row["Gene"]
    print(f"{gene:<8} | {row['Freq']:<15} | {local_freqs[gene]:<10}")

Gene     | cBioPortal %    | Local %   
----------------------------------------
IDH1     | 77.4            | 77.41     
TP53     | 47.8            | 47.15     
ATRX     | 36.7            | 36.74     
CIC      | 21.4            | 21.81     
NOTCH1   | 7.3             | 7.66      
FUBP1    | 8.8             | 8.84      


### Final Somatic Mutation Feature Set

In [13]:
gbm_genes = "TP53", "PTEN", "EGFR", "NF1", "ATRX", "PIK3CA"
lgg_genes = "IDH1", "TP53", "ATRX", "CIC", "NOTCH1", "FUBP1"

master_genes = list(sorted(set(gbm_genes + lgg_genes)))
print(master_genes)

['ATRX', 'CIC', 'EGFR', 'FUBP1', 'IDH1', 'NF1', 'NOTCH1', 'PIK3CA', 'PTEN', 'TP53']


In [14]:
df_output = pd.DataFrame(sorted(master_genes), columns=['Gene'])
df_output.to_csv(r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Final Feature Sets\somatic_mutation_features.tsv", sep='\t', index=False)

## Gene Expression Variables

### Known Glioma Biomarkers

In [15]:
somatic_expression = ['IDH1', 'EGFR', 'PTEN', 'CIC', 'NOTCH1',
                      'TP53', 'FUBP1', 'NF1', 'PIK3CA', 'ATRX']

In [16]:
known_biomarkers = [
    'IDH1', 'IDH2', 'ATRX', 'TP53', 'CDKN2A', 'CDKN2B',
    'CIC', 'FUBP1', 'NOTCH1', 'EGFR',

    'MYB', 'MYBL1', 'BRAF', 'FGFR1', 'ACVR1', 'PDGFRA',
    'EZHIP', 'MYCN', 'NF1', 'TSC1', 'TSC2',
    'PRKCA', 'MN1', 'MET', 'ALK'
]

### Pathway-related Genes

In [17]:
def get_adaptive_genes(filepath, pathway_name, top_n=5):
    print(f"\n{pathway_name}")
    try:
        with open(filepath, "r") as f:
            lines = f.readlines()

        header_idx = next(i for i, line in enumerate(lines) if "Gene Symbol" in line)

        df = pd.read_csv(filepath, sep='\t', skiprows=header_idx)
        df = df.set_index('Gene Symbol')

        pathway_cols = [c for c in df.columns if c not in ['Entrez Gene Id', 'Gene Description']]
        matrix = df[pathway_cols].notnull().astype(int)

        k = [c for c in pathway_cols if 'KEGG' in c.upper()]
        r = [c for c in pathway_cols if 'REACTOME' in c.upper()]
        o = [c for c in pathway_cols if 'KEGG' not in c.upper() and 'REACTOME' not in c.upper()]

        matrix['sources'] = (
            (matrix[k].sum(axis=1) > 0).astype(int) + 
            (matrix[r].sum(axis=1) > 0).astype(int) + 
            (matrix[o].sum(axis=1) > 0).astype(int)
        )
        matrix['total_hits'] = matrix[pathway_cols].sum(axis=1)

        final_df = matrix.sort_values(by=['sources', 'total_hits'], ascending=False)
        consensus_df = final_df[final_df['sources'] >= 2]

        if consensus_df.empty:
            print(f"No overlap found")
            selected = final_df.head(top_n).index.tolist()
        else:
            selected = consensus_df.head(top_n).index.tolist()

        print(f"{selected}")
        return selected

    except Exception as e:
        print(f"Error: {e}")
        return []

In [18]:
pathway_related_genes = []

g2m = get_adaptive_genes(
    r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Gene Expression\Pathways\g2m_overlap.tsv",
    "G2M Cell Cycle"
)
pi3k = get_adaptive_genes(
    r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Gene Expression\Pathways\pi3k_overlap.tsv",
    "PI3K/AKT/mTOR"
)
hypoxia = get_adaptive_genes(
    r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Gene Expression\Pathways\hypoxia_overlap.tsv",
    "Hypoxia"
)


G2M Cell Cycle
['AURKB', 'RPA2', 'MCM5', 'CDC45', 'MCM6']

PI3K/AKT/mTOR
['MAPK1', 'MAPK8', 'MAP3K7', 'MAP2K6', 'MAPK9']

Hypoxia
['ALDOC', 'ALDOB', 'ENO2', 'ENO3', 'PFKP']


In [19]:
emt = get_adaptive_genes(
    r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Gene Expression\Pathways\emt_overlap.tsv",
    "EMT"
)

pathway_related_genes = list(set(g2m + pi3k + hypoxia + emt))
print(f"\n{pathway_related_genes}")


EMT
['COL1A2', 'COL1A1', 'COL5A1', 'COL3A1', 'COL4A2']

['ALDOB', 'COL5A1', 'MAPK1', 'COL1A2', 'ENO3', 'PFKP', 'CDC45', 'COL3A1', 'AURKB', 'MAPK9', 'MCM6', 'RPA2', 'MCM5', 'ALDOC', 'MAPK8', 'COL4A2', 'MAP3K7', 'MAP2K6', 'ENO2', 'COL1A1']


### Published Glioma Expression Signatures

In [20]:
subtype_signature = [
    'ARPC1B', 'S100A11', 'CTSC', 'GLIPR1', 'NNMT', 

    'HN1', 'RAB33A', 'HDAC2', 'MYT1', 'MTSS1',

    'PTPRA', 'ELOVL2', 'MLC1', 'SOX9', 'ARNTL'
]

In [21]:
prognostic_signature = [
    'RAP1B', 'VAV3', 'ZYX', 'COL5A2', 'SPP1', 
    'COL6A1', 'ITGA5', 'SHC1', 'PTEN'
]

### Final Gene Expression Feature Set

In [22]:
raw_lists = [
    somatic_expression, 
    known_biomarkers, 
    pathway_related_genes, 
    subtype_signature, 
    prognostic_signature
]

final_genes = sorted(list(set(
    gene.upper() for sublist in raw_lists for gene in sublist if isinstance(gene, str)
)))
print(f"\nFinal gene count: {len(final_genes)}")
print(final_genes)


Final gene count: 70
['ACVR1', 'ALDOB', 'ALDOC', 'ALK', 'ARNTL', 'ARPC1B', 'ATRX', 'AURKB', 'BRAF', 'CDC45', 'CDKN2A', 'CDKN2B', 'CIC', 'COL1A1', 'COL1A2', 'COL3A1', 'COL4A2', 'COL5A1', 'COL5A2', 'COL6A1', 'CTSC', 'EGFR', 'ELOVL2', 'ENO2', 'ENO3', 'EZHIP', 'FGFR1', 'FUBP1', 'GLIPR1', 'HDAC2', 'HN1', 'IDH1', 'IDH2', 'ITGA5', 'MAP2K6', 'MAP3K7', 'MAPK1', 'MAPK8', 'MAPK9', 'MCM5', 'MCM6', 'MET', 'MLC1', 'MN1', 'MTSS1', 'MYB', 'MYBL1', 'MYCN', 'MYT1', 'NF1', 'NNMT', 'NOTCH1', 'PDGFRA', 'PFKP', 'PIK3CA', 'PRKCA', 'PTEN', 'PTPRA', 'RAB33A', 'RAP1B', 'RPA2', 'S100A11', 'SHC1', 'SOX9', 'SPP1', 'TP53', 'TSC1', 'TSC2', 'VAV3', 'ZYX']


In [23]:
df_output = pd.DataFrame(sorted(final_genes), columns=['Gene'])
df_output.to_csv(r"C:\Users\abbyn\OneDrive\Desktop\Research Project\Biological Feature Selection\Final Feature Sets\gene_expr_features.tsv", sep='\t', index=False)